In [1]:
%pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 2.9 MB/s  0:00:33


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor

In [3]:
default_grid = {
    'depth': [3, 4],
    'learning_rate': [0.03, 0.1, 0.15],
    'random_strength': [1, 5, 10],
    'subsample': [0.8, 1.0],
    'rsm': [0.8, 1.0]
}

In [4]:
# Function to split data into 5 folds
# Returns X_train, X_test, y_train, y_test, and folds
def split_data(df):
    target_1 = ['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-05-01', '2023-06-01', '2023-07-01', '2023-08-01']
    target_2 = ['2023-09-01', '2023-10-01', '2023-11-01', '2023-12-01', '2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01']
    target_3 = ['2024-05-01', '2024-06-01', '2024-07-01', '2024-08-01', '2024-09-01', '2024-10-01', '2024-11-01', '2024-12-01']
    target_4 = ['2025-01-01', '2025-02-01', '2025-03-01', '2025-04-01', '2025-05-01', '2025-06-01', '2025-07-01', '2025-08-01']
    target_5 = ['2025-09-01', '2025-10-01', '2025-11-01', '2025-12-01', '2026-01-01', '2026-02-01', '2026-03-01']

    fold_1 = df[df['PERIOD_BEGIN'].isin(target_1)].copy()
    fold_2 = df[df['PERIOD_BEGIN'].isin(target_2)].copy()
    fold_3 = df[df['PERIOD_BEGIN'].isin(target_3)].copy()
    fold_4 = df[df['PERIOD_BEGIN'].isin(target_4)].copy()
    fold_5 = df[df['PERIOD_BEGIN'].isin(target_5)].copy()

    fold_1['fold_id'] = 0
    fold_2['fold_id'] = 1
    fold_3['fold_id'] = 2
    fold_4['fold_id'] = 3

    train = pd.concat([fold_1, fold_2, fold_3, fold_4], ignore_index=True)
    test = fold_5

    y_train = train.pop('AVG_SALE_TO_LIST')
    X_train = train.drop(columns=['PERIOD_BEGIN', 'PERIOD_END'])
    folds = X_train.pop('fold_id')
    y_test = test.pop('AVG_SALE_TO_LIST')
    X_test = test.drop(columns=['PERIOD_BEGIN', 'PERIOD_END'])

    return X_train, X_test, y_train, y_test, folds

In [5]:
# Parameter tuning function
# Takes csv_filename and param_grid
def parameter_tune(csv_filename, param_grid):
    df = pd.read_csv(csv_filename)

    # Get splits
    X_train, X_test, y_train, y_test, folds = split_data(df)
    X_train['REGION'] = X_train['REGION'].astype(str) # Keep as string because zip codes shouldn't be encoded as numbers

    # Get fold test/valid indices cuz CatBoost needs that
    folding_indices = []
    for fold_id in folds.unique():
        train_idx = np.where(folds != fold_id)[0]
        val_idx = np.where(folds == fold_id)[0]
        folding_indices.append((train_idx, val_idx))

    model = CatBoostRegressor(
        iterations=2000,
        loss_function='MAE',
        eval_metric='MAE',
        early_stopping_rounds=50,
        bootstrap_type='Bernoulli',    # Required to enable subsample
        cat_features=['REGION'],
        max_ctr_complexity=1,
        random_seed=42,
        verbose=False
    )

    # Run grid search over folds
    result = model.grid_search(
        param_grid=param_grid,
        X=X_train,
        y=y_train,
        cv=folding_indices, # Overrides random split cuz we have it split manually
        search_by_train_test_split=False,
        plot=False
    )

    return result


In [ ]:
default_just_zip = parameter_tune('market_zip.csv', default_grid)

Training on fold [0/4]

bestTest = 0.01311644647
bestIteration = 1095

Training on fold [1/4]

bestTest = 0.0122524946
bestIteration = 623

Training on fold [2/4]

bestTest = 0.01099196955
bestIteration = 659

Training on fold [3/4]

bestTest = 0.01074855279
bestIteration = 823

0:	loss: 0.0117818	best: 0.0117818 (0)	total: 13.9s	remaining: 16m 28s
Training on fold [0/4]

bestTest = 0.01306543315
bestIteration = 474

Training on fold [1/4]

bestTest = 0.01223083799
bestIteration = 230

Training on fold [2/4]

bestTest = 0.01104104298
bestIteration = 269

Training on fold [3/4]

bestTest = 0.01080559309
bestIteration = 482

1:	loss: 0.0117979	best: 0.0117818 (0)	total: 21.5s	remaining: 12m 31s
Training on fold [0/4]

bestTest = 0.01312693849
bestIteration = 221

Training on fold [1/4]

bestTest = 0.01225383457
bestIteration = 214

Training on fold [2/4]

bestTest = 0.01107108392
bestIteration = 176

Training on fold [3/4]

bestTest = 0.01082902233
bestIteration = 191

2:	loss: 0.0118327

In [ ]:
default_just_zip['params']

{'subsample': 0.8,
 'rsm': 0.8,
 'random_strength': 10,
 'depth': 3,
 'learning_rate': 0.1}

In [ ]:
just_zip_grid_1 = {
    'depth': [2, 3],
    'learning_rate': [0.08, 0.1, 0.12],
    'random_strength': [10, 15, 25],
    'subsample': [0.65, 0.8],
    'rsm': [0.65, 0.8]
}

just_zip_res_1 = parameter_tune('market_zip.csv', just_zip_grid_1)

Training on fold [0/4]

bestTest = 0.01317915301
bestIteration = 626

Training on fold [1/4]

bestTest = 0.01231812129
bestIteration = 498

Training on fold [2/4]

bestTest = 0.0110613165
bestIteration = 441

Training on fold [3/4]

bestTest = 0.01078771558
bestIteration = 484

0:	loss: 0.0118414	best: 0.0118414 (0)	total: 9.53s	remaining: 11m 16s
Training on fold [0/4]

bestTest = 0.01319607081
bestIteration = 626

Training on fold [1/4]

bestTest = 0.01236577275
bestIteration = 497

Training on fold [2/4]

bestTest = 0.01115551511
bestIteration = 318

Training on fold [3/4]

bestTest = 0.01080317218
bestIteration = 404

1:	loss: 0.0118877	best: 0.0118414 (0)	total: 18.3s	remaining: 10m 40s
Training on fold [0/4]

bestTest = 0.01316977273
bestIteration = 478

Training on fold [1/4]

bestTest = 0.01237122324
bestIteration = 202

Training on fold [2/4]

bestTest = 0.0110538286
bestIteration = 378

Training on fold [3/4]

bestTest = 0.01080884053
bestIteration = 494

2:	loss: 0.0118598	b

In [ ]:
print(just_zip_res_1['params'])
just_zip_iter = np.argmin(just_zip_res_1['cv_results']['test-MAE-mean']) + 1
just_zip_best_iter = (519 + 282 + 305 + 291) / 4
print(just_zip_iter)
print(just_zip_best_iter)

{'subsample': 0.8, 'rsm': 0.65, 'random_strength': 15, 'depth': 3, 'learning_rate': 0.08}
520
349.25


In [ ]:
default_all = parameter_tune('market_zip_dem.csv', default_grid)

Training on fold [0/4]

bestTest = 0.01307798035
bestIteration = 1354

Training on fold [1/4]

bestTest = 0.01191581429
bestIteration = 1246

Training on fold [2/4]

bestTest = 0.01071815612
bestIteration = 1440

Training on fold [3/4]

bestTest = 0.01050366969
bestIteration = 825

0:	loss: 0.0115558	best: 0.0115558 (0)	total: 27.2s	remaining: 32m 12s
Training on fold [0/4]

bestTest = 0.01297605011
bestIteration = 590

Training on fold [1/4]

bestTest = 0.0119304027
bestIteration = 588

Training on fold [2/4]

bestTest = 0.01074239664
bestIteration = 315

Training on fold [3/4]

bestTest = 0.01054460512
bestIteration = 406

1:	loss: 0.0115615	best: 0.0115558 (0)	total: 38.8s	remaining: 22m 38s
Training on fold [0/4]

bestTest = 0.01306066626
bestIteration = 364

Training on fold [1/4]

bestTest = 0.01196508728
bestIteration = 322

Training on fold [2/4]

bestTest = 0.01080683833
bestIteration = 311

Training on fold [3/4]

bestTest = 0.01058977312
bestIteration = 264

2:	loss: 0.01162

In [ ]:
default_all['params']

{'subsample': 1,
 'rsm': 0.8,
 'random_strength': 5,
 'depth': 3,
 'learning_rate': 0.03}

In [6]:
all_grid_1 = {
    'depth': [3, 4],
    'learning_rate': [0.02, 0.03, 0.04],
    'random_strength': [3, 5, 8],
    'subsample': [0.9, 1.0],
    'rsm': [0.65, 0.8, 0.9]
}

all_res_1 = parameter_tune('market_zip_dem.csv', all_grid_1)

Training on fold [0/4]

bestTest = 0.01308562787
bestIteration = 1994

Training on fold [1/4]

bestTest = 0.01192426579
bestIteration = 1858

Training on fold [2/4]

bestTest = 0.01071514522
bestIteration = 1999

Training on fold [3/4]

bestTest = 0.01045062024
bestIteration = 1300

0:	loss: 0.0115447	best: 0.0115447 (0)	total: 36.7s	remaining: 1h 5m 30s
Training on fold [0/4]

bestTest = 0.01300685596
bestIteration = 1999

Training on fold [1/4]

bestTest = 0.01199042301
bestIteration = 991

Training on fold [2/4]

bestTest = 0.01072452215
bestIteration = 1340

Training on fold [3/4]

bestTest = 0.01045224101
bestIteration = 841

1:	loss: 0.0115448	best: 0.0115447 (0)	total: 1m 1s	remaining: 54m 37s
Training on fold [0/4]

bestTest = 0.01303190113
bestIteration = 1364

Training on fold [1/4]

bestTest = 0.01194154633
bestIteration = 1446

Training on fold [2/4]

bestTest = 0.01080246063
bestIteration = 745

Training on fold [3/4]

bestTest = 0.01045664574
bestIteration = 869

2:	loss:

In [7]:
print(all_res_1['params'])
all_1_iter = np.argmin(all_res_1['cv_results']['test-MAE-mean']) + 1
all_1_best_iter = (1537 + 1075 + 1007 + 741) / 4
print(all_1_iter)
print(all_1_best_iter)

{'subsample': 0.9, 'rsm': 0.65, 'random_strength': 5, 'depth': 4, 'learning_rate': 0.03}
1538
1090.0
